# 第 12 章: 正則化とモデル選択の探索と可視化

alpha を細かく変えたときの係数と決定係数の変化、ラッソ回帰で 0 になる係数の数、分け方による結果の違いを確認する。

Polyglot Notebooks（.NET Interactive）は 2026 年に廃止された。この Notebook は `Microsoft.dotnet-interactive` 1.0.712001 と Plotly.NET.Interactive 5.0.0 で動作を確かめている。
先に `dotnet build` で `apps/fsharp/` のライブラリをビルドしておく。

In [ ]:
#r "nuget: FSharp.Data, 8.2.0"
#r "nuget: Microsoft.ML, 5.0.0"
#r "nuget: Plotly.NET, 5.1.0"
#r "nuget: Plotly.NET.Interactive, 5.0.0"
#r "../src/MachineLearning/bin/Debug/net10.0/MachineLearning.dll"

In [ ]:
open System.IO
open Plotly.NET
open MachineLearning.Dataset
open MachineLearning.Chapter07.RegressionMetrics
open MachineLearning.Chapter12.Boston
open MachineLearning.Chapter12.Regularization

let csvFile = Path.Combine(dataDir (), "Boston.csv")
let boston = prepareBoston csvFile 0.3 0.3 0
let alphas = [ for k in -2.0 .. 0.25 .. 3.0 -> 10.0 ** k ]

## alpha と係数の変化

In [ ]:
let models = alphas |> List.map (fun alpha -> fitRidge alpha boston.XTrain boston.TTrain)

boston.FeatureNames
|> List.mapi (fun i name ->
    Chart.Line(x = (alphas |> List.map log10), y = (models |> List.map (fun m -> m.Coefficients[i])), Name = name))
|> Chart.combine
|> Chart.withTitle "alpha と係数の変化"
|> Chart.withXAxisStyle "log10(alpha)"
|> Chart.withYAxisStyle "係数"

## alpha と決定係数

In [ ]:
let experiments =
    runRidgeExperiments alphas boston.XTrain boston.TTrain boston.XValid boston.TValid

[
    Chart.Line(x = (alphas |> List.map log10), y = (experiments |> List.map (fun e -> e.TrainScore)), Name = "訓練 R²")
    Chart.Line(x = (alphas |> List.map log10), y = (experiments |> List.map (fun e -> e.ValidationScore)), Name = "検証 R²")
]
|> Chart.combine
|> Chart.withTitle "alpha と決定係数"
|> Chart.withXAxisStyle "log10(alpha)"
|> Chart.withYAxisStyle "決定係数"

## ラッソ回帰で 0 になる係数

In [ ]:
let lassoAlphas = [ 0.05; 0.1; 0.2; 0.5; 1.0; 2.0; 5.0 ]

let zeroCounts =
    lassoAlphas
    |> List.map (fun alpha ->
        let model = fitLasso alpha boston.XTrain boston.TTrain
        alpha, (zeroCoefficientNames model.Coefficients boston.FeatureNames).Length)

Chart.Column(values = (zeroCounts |> List.map snd), Keys = (zeroCounts |> List.map (fst >> string)))
|> Chart.withTitle "ラッソ回帰の alpha と 0 になった係数の数"
|> Chart.withXAxisStyle "alpha"
|> Chart.withYAxisStyle "0 になった係数の数"

In [ ]:
zeroCounts |> List.map (fun (alpha, count) -> {| alpha = alpha; 係数が0の数 = count |}) |> List.toArray

## 分け方による結果の違い

In [ ]:
[ 0..19 ]
|> List.map (fun seed ->
    let d = prepareBoston csvFile 0.3 0.3 seed
    let score alpha = fitRidge alpha d.XTrain d.TTrain |> fun m -> predictRegularized m d.XTest |> r2Score d.TTest
    {| シード = seed; 線形回帰 = score 0.0; リッジ回帰 = score 10.0 |})
|> List.toArray